- This is the code for the integration of datasets from the ScPCA
- Using the python environment singlec
- Script was slightly modified to process GEO data as it was in different file formats 

In [ ]:
import os
from pathlib import Path

# Override with: export B_CELL_ATLAS_DATA=/path/to/your/data
DATA_ROOT = Path(os.environ.get('B_CELL_ATLAS_DATA', 'data'))

# Each folder in the python_data folder has multiple directory with the name of the dataset. These directories contain the unfiltered RNA h5ad files
!ls python_data/scpc8/

In [ ]:
# Import statements
import scanpy as sc
import pandas as pd
import os
import gc

In [ ]:

data_dir = '/home/sbadol2/python_data/scpc8'
with open('/home/sbadol2/python_data/scpc8/combined.txt', 'r') as file:
    h5ad_files = [line.strip() for line in file]

adatas = []

for file in h5ad_files:
    file_path = os.path.join(data_dir, file)
    if os.path.exists(file_path):
        adata = sc.read_h5ad(file_path, backed='r') # using backed to load bigger datasets
        adatas.append(adata)  
    else:
        print(f"File {file_path} doesn't exist")

if adatas:
    merged_adata = sc.concat(adatas)
    merged_adata.var_names_make_unique()
    merged_adata.obs_names_make_unique()

In [ ]:
merged_adata

In [ ]:
merged_adata.obs

In [ ]:

# Filtering and QC
sc.pp.filter_cells(merged_adata, min_counts=1000)  
sc.pp.filter_cells(merged_adata, min_genes=200)   

# Filter mitochondrial percentage < 20% - for ScPCA samples, subsets_mito_percent was used
merged_adata = merged_adata[merged_adata.obs['subsets_mito_percent'] < 20]

merged_adata

In [ ]:
# Clean up unwanted metadata 
for n in ['mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts']:
    if n in merged_adata.var:
        del merged_adata.var[n]
        gc.collect()

for x in ['n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt']:
    if x in merged_adata.obs:
        del merged_adata.obs[x]
        gc.collect()

In [ ]:
merged_adata

In [ ]:
merged_adata.obs

In [ ]:
# included columns
exclude_columns = ['sample_id', 'scpca_project_id', 'diagnosis', 'suspension_type']

for x in list(merged_adata.obs.columns):  
    if x not in exclude_columns:  
        del merged_adata.obs[x]  
        gc.collect()  

In [ ]:
merged_adata.obs

In [ ]:
# Save the merged AnnData object to an .h5ad file
merged_adata.write("/home/sbadol2/python_data/scpc8/scpc8_4.h5ad")

In [ ]:
gc.collect()

In [ ]:
import anndata as ad

# Load the two h5ad files

# file1 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905416_MUV39.h5ad"))
# file2 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905417_MUV41.h5ad"))
# file3 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905418_MUV44.h5ad"))
# file4 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905419_SJ17.h5ad"))
# file5 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905420_SJ99.h5ad"))
# file6 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905421_SJ129.h5ad"))
# file7 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905422_SJ217.h5ad"))
# file8 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905423_SJ454.h5ad"))
# file9 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905424_SJ516.h5ad"))
# file10 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905425_SJ577.h5ad"))
file1 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905426_SJ617.h5ad"))
file2 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905427_SJ625.h5ad"))
file3 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905428_SJ723.h5ad"))
file4 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905429_SJ917.h5ad"))
file5 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905430_SJ970.h5ad"))
file6 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905431_DMB006.h5ad"))
file7 =  ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905432_Icb1299.h5ad"))
file8 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905433_Icb1572.h5ad"))
file9 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905434_Med114FH.h5ad"))
file10 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905435_Med2112FH.h5ad"))
# file1 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905436_Med211FH.h5ad"))
# file2 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905437_Med2312FH.h5ad"))
# file3 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905438_Med411FH.h5ad"))
# file4 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905439_RCMB18.h5ad"))
# file5 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905440_RCMB20.h5ad"))
# file6 = ad.read_h5ad(str(DATA_ROOT / "GSE119926/GSM3905441_RCMB24.h5ad"))



# Concatenate the files along the observation axis (default)
merged = ad.concat([ merged, file1, file2, file3, file4, file5, file6, file7, file8, file9, file10], join="outer")
merged


In [ ]:
merged

In [ ]:
merged.obs

In [ ]:
# Identify mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('MT-')

# Calculate percentage of mitochondrial counts per cell
adata.obs['percent_mito'] = (adata[:, adata.var['mt']].X.sum(axis=1) / adata.X.sum(axis=1)) * 100

In [ ]:
#Filtering and QC
sc.pp.filter_cells(merged, min_counts=1000) 
sc.pp.filter_cells(merged, min_genes=200) 

adata = adata[adata.obs['percent_mito'] < 20]

merged

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_qc_metrics(merged):
# Create figure\n",
    plt.figure(figsize=(12, 6))

    # Plot genes/features detected
    plt.subplot(1, 3, 1)
    sns.violinplot(y=merged.obs['detected'], color="skyblue")
    plt.title("Genes/Features Detected")
    plt.ylabel("Count")

    #Plot total counts\n",
    plt.subplot(1, 3, 2)
    sns.violinplot(y=merged.obs['total'], color="lightgreen")
    plt.title("Total Counts")
    plt.ylabel("Count")

    # Plot mitochondrial percentage\n",
    plt.subplot(1, 3, 3)
    sns.violinplot(y=merged.obs['subsets_mito_percent'], color="salmon")
    plt.title("Mitochondrial Percentage")
    plt.ylabel("Percentage")
    
    plt.tight_layout()
    plt.show()


plot_qc_metrics(merged)

In [ ]:
# Add dataset_id column to obs
merged.obs['dataset_id'] = 'SCPC12'

In [ ]:
# included columns\n",
exclude_columns = ['sample_id', 'scpca_project_id', 'diagnosis', 'suspension_type', 'dataset_id']
for x in list(merged.obs.columns):
    if x not in exclude_columns:
        del merged.obs[x] 
   

# Make observation names unique
merged.obs_names_make_unique()

# Make variable names unique (if needed)
merged.var_names_make_unique()

In [ ]:
merged

In [ ]:
merged.obs

In [ ]:
merged.write(str(DATA_ROOT / "external" / "Python_data/scpc12/scpc12_merged.h5ad"))

In [ ]:
gc.collect()